In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

#Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
data = df.copy()
data = data.drop(columns=['Order_ID'])
data

In [ ]:
data.info()

In [ ]:
# Task 2: Write your code here:

def fill_missing_values(df, target_col=None, num_strategy="mean"):

    df = df.copy()

    # Numerical columns (except target)
    numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns
    if target_col in numerical_cols:
        numerical_cols = numerical_cols.drop(target_col)

    # Categorical columns
    categorical_cols = df.select_dtypes(include=["object"]).columns

    # Fill numerical columns
    for col in numerical_cols:
        if df[col].isnull().sum() > 0:
            if num_strategy == "mean":
                df[col].fillna(df[col].mean(), inplace=True)
            elif num_strategy == "median":
                df[col].fillna(df[col].median(), inplace=True)

    # Fill categorical columns
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mode()[0], inplace=True)

    # Drop rows where target is missing
    if target_col is not None:
        df = df.dropna(subset=[target_col])

    return df
data = fill_missing_values(data)
data.info()


In [ ]:
data.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = data.select_dtypes(include=["object"]).columns

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  data[col] = le.fit_transform(data[col])
  label_encoders[col] = le

data


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = data.columns.drop("Delivery_Time")

scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])

In [ ]:
# Task 6: Write your code here:
# not needed cuz its linear reg

In [ ]:
# Task 1: Write your code here:
X = data.drop("Delivery_Time", axis=1).astype(float)
y = data['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
n_splits = 5

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_mae = []


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
n_estimators = 200
max_depth = None
model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Metrics
  mae = mean_absolute_error(y_test, y_pred)
  print(f"  MAE:  {mae:.4f}")

  # Store results
  lr_mae.append(mae)

y_pred

In [ ]:
# Task 1: Write your code here:
def plot_trained_feature_importance(model, X, model_type="Randomforest"):
  coef_ = model.feature_importances_

  plt.figure(figsize=(10,5))
  plt.bar(X.columns, coef_)
  plt.xticks(rotation=45)
  plt.title(f"Feature Importance after Training ({model_type})")
  plt.show()

plot_trained_feature_importance(model, X)

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10,5))
plt.hist(y_pred, edgecolor='black')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Task Bonus: Write your code here: